### Checklist for submission

It is extremely important to make sure that:

1. Everything runs as expected (no bugs when running cells);
2. The output from each cell corresponds to its code (don't change any cell's contents without rerunning it afterwards);
3. All outputs are present (don't delete any of the outputs);
4. Fill in all the places that say `# YOUR CODE HERE`, or "**Your answer:** (fill in here)".
5. Never copy/paste any notebook cells. Inserting new cells is allowed, but it should not be necessary.
6. The notebook contains some hidden metadata which is important during our grading process. **Make sure not to corrupt any of this metadata!** The metadata may for example be corrupted if you copy/paste any notebook cells, or if you perform an unsuccessful git merge / git pull. It may also be pruned completely if using Google Colab, so watch out for this. Searching for "nbgrader" when opening the notebook in a text editor should take you to the important metadata entries.
7. Although we will try our very best to avoid this, it may happen that bugs are found after an assignment is released, and that we will push an updated version of the assignment to GitHub. If this happens, it is important that you update to the new version, while making sure the notebook metadata is properly updated as well. The safest way to make sure nothing gets messed up is to start from scratch on a clean updated version of the notebook, copy/pasting your code from the cells of the previous version into the cells of the new version.
8. If you need to have multiple parallel versions of this notebook, make sure not to move them to another directory.
9. Although not forced to work exclusively in the course `conda` environment, you need to make sure that the notebook will run in that environment, i.e. that you have not added any additional dependencies.

Failing to meet any of these requirements might lead to the submission not being validated meaning no feedback will be provided.

We advise you to perform the following steps before submission to ensure that requirements 1, 2, and 3 are always met: **Restart the kernel** (in the menubar, select Kernel$\rightarrow$Restart) and then **run all cells** (in the menubar, select Cell$\rightarrow$Run All). This might require a bit of time, so plan ahead for this. Finally press the "Save and Checkout" button before handing in, to make sure that all your changes are saved to this .ipynb file.

### Fill in name of notebook file
This might seem silly, but the version check below needs to know the filename of the current notebook, which is not trivial to find out programmatically.

You might want to have several parallel versions of the notebook, and it is fine to rename the notebook as long as it stays in the same directory. **However**, if you do rename it, you also need to update its own filename below:

In [ ]:
nb_fname = "XXX.ipynb"

assert (nb_fname != "XXX.ipynb"), "Make sure to fill in the nb_fname variable above!"

### Fill in group number and member names:

In [ ]:
NAME1 = ""
NAME2 = ""
GROUP = ""

### Check Python version

In [ ]:
from platform import python_version_tuple

assert (
    python_version_tuple()[:2] == ("3", "11")
), "You are not running Python 3.11. Make sure to run Python through the course Conda environment."

### Check that notebook server has access to all required resources, and that notebook has not moved

In [ ]:
import os

nb_dirname = os.path.abspath("")
assignment_name = os.path.basename(nb_dirname)
assert assignment_name in [
    "HA1",
    "HA2",
    "HA3",
    "HA4",
], "[ERROR] The notebook appears to have been moved from its original directory"

### Verify correct nb_fname

In [ ]:
from IPython.display import HTML, display

try:
    display(
        HTML(
            r'<script>if("{nb_fname}" != IPython.notebook.notebook_name) {{ alert("You have filled in nb_fname = \"{nb_fname}\", but this does not seem to match the notebook filename \"" + IPython.notebook.notebook_name + "\"."); }}</script>'.format(
                nb_fname=nb_fname
            )
        )
    )
except NameError:
    assert False, "Make sure to fill in the nb_fname variable above!"

### Verify that your notebook is up-to-date and not corrupted in any way

In [ ]:
import sys

sys.path.append("..")
from ha_utils import check_notebook_uptodate_and_not_corrupted

check_notebook_uptodate_and_not_corrupted(nb_dirname, nb_fname)

# HA4: Part 1 - Recurrent Neural Networks

This is the first part of a two-part assignment.
In it, you will implement a vanilla RNN module from scratch using PyTorch to perform the task of predicting the nationality of a given name. 

For efficient use of GPU hours, you can do all development up until the training on a CPU. You can then run the training on a GPU for faster training times.

**IMPORTANT NOTES:**
Some cells contain compiled tests. For them to work properly, make sure to keep function and class names.
Similarly to HA1, HA2, and HA3, some questions in this notebook will not be graded in detail.
These are marked as \[extra\].
It's still recommended to answer them since related questions may appear in the Inspera test.
You wont need to train neural networks in Inspera test.

# Task 1 - Predicting Nationalities

In this task, you will create an RNN module using PyTorch, and train it to predict the nationality of a given input name. Your RNN will process each character of an input name at a time, and at the last character output a probability mass function over the possible countries.

The data for this task is present in the file `data/nam_dict.txt`. After parsing it we will have around 44k names from 41 different countries.

---
## 1.0 Imports

Import dependencies

In [ ]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support
from torch import nn
from torch.utils.data import DataLoader, TensorDataset, random_split

from utils.load_names import get_data_from_file
from utils.tests import ha4_part1_tests

---
## 1.1 Loading the data

We'll start by loading the data from the `nam_dict.txt` file, using the provided method `get_data_from_file`.

In [ ]:
names_dict = get_data_from_file("data/nam_dict.txt")

The newly-created variable `names_dict` is a dictionary where each country is a key, and the value is the corresponding list of names from that country.

In [ ]:
names_dict.keys()

In [ ]:
names_dict["Greece"][:10]

Let's look at the dataset in more detail. Plot the number of names in each country using a [bar plot](https://matplotlib.org/3.1.1/api/_as_gen/matplotlib.pyplot.bar.html).

*Hints*:

- You can put labels in the x-axis using the method [`xticks`](https://matplotlib.org/3.1.1/api/_as_gen/matplotlib.pyplot.xticks.html).
- The `xticks` method accepts a keyword argument named `rotation`.

In [ ]:
%matplotlib inline

# YOUR CODE HERE

\[extra\]: Is this dataset balanced? If not, which are the top-5 countries with most names?

**Your answer:** (fill in here)

---
## 1.2 Pre-processing

We will train an RNN which takes as input one character at each time-step. In order to do so, we will encode every character in the input names into a one-hot representation that is more suited to be fed to neural networks. 

The first thing we need to do is to decide the size of the alphabet used for the one-hot encoding. To do so, let's take a look at all the characters used in the dataset. A simple way to do so is putting all names in the same string and then using a `set` to get the unique values. We'll treat upper-case and lower-case characters as being the same.

In [ ]:
temp = "".join(name.lower()
               for names in names_dict.values()
               for name in names)
charset = sorted(set(temp))
print(f"{len(charset)} characters:")
charset

We will now create a function that takes as input a name and the set of characters, and outputs a `(name length) x (charset length)` tensor, with each row being a one-hot encoding of the corresponding character. Complete the function below.

In [ ]:
def name2tensor(name, charset):
    name = name.lower()

    tensor = None
    # YOUR CODE HERE

    return tensor

Run the following cell without changing anything to test your implementation of the above method. 

In [ ]:
ha4_part1_tests.test_name2_tensor(name2tensor)

Let us look at an example tensor for a name.

In [ ]:
name2tensor("Ali", charset)

\[extra\]: Why does the above output tensor contain a 1 at position (0,3)?

**Your answer:** (fill in here)

\[extra\]: What would if you try encoding the string `Lol@` using this function? Why?

**Your answer:** (fill in here)

It's definitely possible to use the tensors created with `name2tensor` to train our RNN, but in that case we would only be able to conveniently perform forward-propagation using one sample at a time.

Instead, we want to batch samples together and parallelize the forward-propagation step. To do so, all samples must have the same dimensions (right now every sample has dimension `(name length) x (charset length)`, and the length of the names varies). Modify the function `name2padded_tensor` to achieve this by padding the start of the tensor with zeros, so that the output tensor always has length `target_len x (charset length)`.

*Hint*:

- To make it easier to catch bugs in the next parts of the assignment, make sure to raise an exception if the `target_len` is smaller than the length of the input word.

In [ ]:
def name2padded_tensor(name, target_len, charset):
    tensor = name2tensor(name, charset)

    # YOUR CODE HERE

    return tensor

In [ ]:
ha4_part1_tests.test_name2padded_tensor(name2padded_tensor, charset)

What should be the target length for our case in this dataset? Compute it (i.e. don't hardcode it) in the cell below and save it in a variable named `target_len`, which will be used from here on

In [ ]:
# YOUR CODE HERE

In [ ]:
ha4_part1_tests.test_target_len(target_len)

Now we can go through the entire dataset, encode each of the names, and save all of it in a tensor. We'll do the same with the ground-truth labels for the countries of each name.

In [ ]:
# Create the lists that will hold the names and labels
xs_list = []
ys_list = []

for country_idx, names in enumerate(names_dict.values()):
    # Stack all padded one-hot names
    # Shape: (len(names_of_country), target_len, len(charset))
    country_xs = torch.stack([name2padded_tensor(name, target_len, charset)
                              for name in names])

    # Goal is to predict nationality
    country_ys = torch.full(size=(len(names),),
                            fill_value=country_idx,
                            dtype=torch.long)
    
    xs_list.append(country_xs)
    ys_list.append(country_ys)

# concatenate the lists
xs = torch.cat(xs_list, dim=0)  # (num_names, target_len, len(charset))
ys = torch.cat(ys_list, dim=0)  # (num_names,)

In [ ]:
xs.shape

In [ ]:
ys.shape

Now we can create a dataset with these tensors.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using {device}")
dataset = TensorDataset(xs.to(device), ys.to(device))

And split it in training, validation, and test sets.

In [ ]:
val_ratio = 0.15
test_ratio = 0.15
train_ratio = 1 - val_ratio - test_ratio

train_dataset, val_dataset, test_dataset = random_split(
    dataset, [train_ratio, val_ratio, test_ratio]
)

---
### 1.3 Defining the optimization

Now we will define the model to be trained, that is, a single layer RNN. As usual, we will create a class for the model and inherit from `nn.Module`. Define in the next cell a class for an RNN that follows the equations:

$$ h_t = \text{tanh}(W_h x_t + U_h h_{t-1} + b_h) $$
$$ y_t = \log \Big(\text{Softmax}(W_y h_t + b_y)\Big)~,$$

where $h_t$ is the current hidden-state, $h_{t-1}$ is the previous hidden state, $x_t$ is the input, $y_t$ is the output, and $W_h, U_h, b_h, W_y, b_y$ are the trainable parts of the RNN. As explained before, we will use this RNN to classify names, inputting one character at each step and using the last output as the prediction for a distribution over probable countries for the name.

*Hints:*

- As you're probably aware by now, PyTorch [has a layer](https://pytorch.org/docs/stable/nn.html#logsoftmax) that already applies the log and the softmax in one pass.
- If you get errors related to size mismatches when computing the forward-propagation in your model, try reading the documentation for the specific module where the problem is occurring (e.g. [`nn.Linear`](https://pytorch.org/docs/stable/nn.html#linear)), especially the part about the expected shape of inputs and outputs.


In [ ]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        """
        Inputs:
            input_size     - Dimensionality of the input of this RNN.
            hidden_size    - Dimensionality of the hidden state.
            output_size    - Dimensionality of the output of this RNN.
        """

        super(RNN, self).__init__()
        self.hidden_size = hidden_size

        # YOUR CODE HERE

    def forward(self, x, h):
        """
        Runs the RNN module for one time-step.
        Inputs:
            x              - Batch of one-hot encoded characters.
                             Tensor of size (BATCH SIZE x CHARSET LENGTH).
            h              - Batch of hidden states at previous time-step.
                             Tensor of size (BATCH SIZE x `hidden_size`).

        Returns:
            y              - Batch of outputs.
                             Tensor of size (BATCH SIZE x NUMBER OF COUNTRIES).
            new_h          - Batch of hidden states at current time-step.
                             Tensor of size (BATCH SIZE x `hidden_size`).
        """

        # YOUR CODE HERE

        return y, new_h

Run the following cell to test whether the shape of your outputs are correct. Note that this *only* checks the shapes and some basic ranges of outputs, not the content (one way you can check the actual computations of your network is to `print` the tensors as they pass through the layers, in order to check that the computations do what you expected).

In [ ]:
ha4_part1_tests.test_model_output(RNN, device, names_dict, charset)

\[extra\]: Suppose we now create an RNN using the following call: `RNN(x, y, z)`. How many parameters would this model have in total?

**Your answer:** (fill in here)

We will use the Adam optimizer:

In [ ]:
from torch.optim import Adam

And, since this is a classification problem we will use the negative log-likelihood loss ([`NLLLoss`](https://pytorch.org/docs/stable/generated/torch.nn.NLLLoss.html#torch.nn.NLLLoss)) to train the model. We should account for the fact that our dataset is imbalanced, and one way of dealing with this problem is to assign different weights to each class in our problem. More specifically, we want to give higher weights to examples belonging to the less frequent classes and lower weights to the more frequent ones.

In [ ]:
# Compute weights for each class
n_names_for_each_country = [len(v) for v in names_dict.values()]
ns = torch.tensor(n_names_for_each_country, dtype=torch.float)
w = 1.0 / ns
w = w / w.sum()

 Declare the appropriate weight criterion using the weights defined above.

In [ ]:
# Define loss

criterion = None
# YOUR CODE HERE

In [ ]:
ha4_part1_tests.test_criterion(criterion, device)

\[extra\]: Why is it a problem that the dataset is imbalanced? What would be the consequence of not dealing with this problem?

**Your answer:** (fill in here)

\[extra\]: Why does assigning different weights to each class solve this problem?

**Your answer:** (fill in here)

\[extra\]: Is there any situation where *we would want* to introduce class imbalances in your dataset? How could we create such a class imbalance in a balanced dataset (without changing the dataset itself)?

**Your answer:** (fill in here)

---
### 1.4 Helper functions for training

Now that we defined the optimization problem, we can start creating the code to actually solve it. The first thing that we need is a function that, given a batch of samples, computes the output of the model for every sample and the average loss across all samples. 

Complete the function `batch_forward_prop` to produce this behavior.

*Hints*:

- Initialize the hidden state to a zero Tensor for the first forward-propagation.
- If you get errors related to size mismatches when computing the forward-propagation in your model, try reading the documentation for the specific module where the problem is occurring (e.g., [`nn.Linear`](https://pytorch.org/docs/stable/nn.html#linear)), especially the part about the expected shape of inputs and outputs.
- If you get errors related to size mismatches when computing the loss, try reading the [documentation for it](https://pytorch.org/docs/stable/nn.html#nllloss), especially the part about the expected shape of inputs and outputs.

In [ ]:
def batch_forward_prop(rnn, xs, ys):
    """
    Inputs:
        rnn            - The RNN model, instance of the RNN class.
        xs             - Batch of input words.
                         Tensor of shape (BATCH SIZE x `target_len` x CHARSET LENGTH).
        ys             - Batch of ground-truth labels.
                         Tensor of  shape (BATCH_SIZE).

    Returns:
        output         - Output computed at the last character position in the batch.
                         Tensor of shape (BATCH_SIZE x NUMBER OF COUNTRIES).
        loss           - Value of the average loss across the predictions for this batch, later used for back-propagation.
                         Tensor with a single element inside.
    """

    # YOUR CODE HERE

    return output, loss

Run the following cell to test whether the shape of your outputs are correct. Note that this *only* checks the shapes, not the content.

In [ ]:
ha4_part1_tests.test_batch_forward_rnn(
    name2tensor_fn=name2tensor,
    charset=charset,
    RNN_model=RNN,
    device=device,
    batch_forward_prop_fn=batch_forward_prop,
    names_dict=names_dict,
)

Using this function, it's straightforward to train on a single batch of data:

In [ ]:
def train_batch(rnn, xs, ys, optimizer):
    # Compute the output for all samples in the batch and the average loss
    output, loss = batch_forward_prop(rnn, xs, ys)

    # Zero gradients before computing backward-propagation
    optimizer.zero_grad()

    # Backward-propagation
    loss.backward()

    # Clip the gradient norm (optional, helps to stabilize training)
    nn.utils.clip_grad_norm_(rnn.parameters(), 2)

    # Perform one step of optimization
    optimizer.step()

    return output, loss

We will also need a function for computing metrics of interest in the validation set. In this case, we will plot both the loss and the F1-score in the validation set. The F1-score of a classifier is the [harmonic mean](https://en.wikipedia.org/wiki/Harmonic_mean) of its [precision and recall](https://en.wikipedia.org/wiki/Precision_and_recall), two metrics that are more useful than accuracy for imbalanced datasets. [This blog post](https://towardsdatascience.com/accuracy-precision-recall-or-f1-331fb37c5cb9) can help you to familiarize yourself with these new metrics.

In [ ]:
def compute_metrics_on_validation_set(rnn, val_dataset):
    # Get all the input and labels in the validation set.
    x_val, y_val = val_dataset[:]

    # Perform forward-prop in the entire validation set, with autograd disabled
    with torch.no_grad():
        val_output, val_loss = batch_forward_prop(rnn, x_val, y_val)

    # Get numpy arrays for the true labels and the predictions
    y_true = y_val.cpu().numpy()
    y_pred = val_output.argmax(dim=1).cpu().numpy()

    # Compute precision, recall, and F-score
    precision, recall, fscore, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro"
    )

    return val_loss, precision, recall, fscore

Lastly, we define a function for plotting the metrics in real-time.

In [ ]:
def plot_metrics(fig, ax, ns, train_losses, train_fscores, val_losses, val_fscores):
    # Plot losses
    ax[0].clear()
    ax[0].plot(ns, train_losses)
    ax[0].plot(ns, val_losses)
    ax[0].set_title("Loss")
    ax[0].legend(["Train", "Validation"])
    ax[0].set_xlabel("Number of trained batches")
    ax[0].grid()

    # Plot F1-scores
    ax[1].clear()
    ax[1].plot(ns, train_fscores)
    ax[1].plot(ns, val_fscores)
    ax[1].plot(ns, [0.3] * len(ns), "k--")
    ax[1].set_title("Macro F1-score")
    ax[1].legend(["Train", "Validation", "F1-score threshold"])
    ax[1].set_xlabel("Number of trained batches")
    ax[1].grid()

    fig.canvas.draw()

With these helper functions, the `train` function becomes:

In [ ]:
%matplotlib notebook

def train(rnn, n_epochs, learning_rate, batch_size, train_dataset, val_dataset):
    # Setup the figure for plotting progress during training
    fig, ax = plt.subplots(ncols=2, figsize=(12, 4))
    plt.ion()
    plot_interval = 100

    # Create arrays to average training metrics across batches
    preds = []
    labels = []
    losses = []

    # Create dictionaries to hold the computed metrics in
    train_data = {"losses": [], "fscores": []}
    val_data = {"losses": [], "fscores": []}

    optimizer = Adam(rnn.parameters(), lr=learning_rate)
    train_data_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True, drop_last=True
    )
    batch_idxs = []

    # Training loop
    i_batch = 0
    for n in range(n_epochs):
        for i, (x_batch, y_batch) in enumerate(train_data_loader):
            i_batch += 1

            # Compute loss and outputs
            output, loss = train_batch(rnn, x_batch, y_batch, optimizer)

            # Aggregate for later averaging
            preds += output.argmax(dim=1).cpu().tolist()
            labels += y_batch.cpu().tolist()
            losses.append(loss)

            # Compute metrics and plot after every `plot_interval` batches
            if i % plot_interval == 0:
                val_loss, _, _, val_fscore = compute_metrics_on_validation_set(
                    rnn, val_dataset
                )
                train_fscore = precision_recall_fscore_support(
                    labels, preds, average="macro"
                )[2]

                val_data["losses"].append(val_loss.cpu())
                val_data["fscores"].append(val_fscore)
                train_data["losses"].append((sum(losses) / len(losses)).item())
                train_data["fscores"].append(train_fscore)
                batch_idxs.append(i_batch)

                preds = []
                labels = []
                losses = []

                plot_metrics(
                    fig,
                    ax,
                    batch_idxs,
                    train_data["losses"],
                    train_data["fscores"],
                    val_data["losses"],
                    val_data["fscores"],
                )

        print(f"Last step validation loss: {val_loss:.3f}, F1-score: {val_fscore:.3f}")

Make sure that you completely understand the `train` function before proceeding (e.g. that we aggregate predictions from different batches and compute metrics only at a certain interval, etc).

---
### 1.5 Training!

Now we're ready to train the network! Create an RNN and use the `train` function to train it.

*Hints*:

- Tuning the hyper-parameters (number of hidden units, learning rate, batch size, etc) will take some trial-and-error. Try simple things first, and then once you manage to train them, start scaling up. Also, have in mind the bias-variance tradeoff mentioned in the lectures.
- When tuning the learning rate, focus first on being able to decrease the training loss. Keep decreasing the learning rate until that starts happening.
- You should use a GPU for fast training times. Using a CPU, training a network can take ~5-10 minutes with suitable hyper-parameters. With a GPU training takes ~1 minute with the same hyper-parameters.

**Note:** If the interactive plots don't work, try to downgrade the notebook as following:

```!pip install notebook==6.5 traitlets==5.9```

In [ ]:
# YOUR CODE HERE

In [ ]:
_, _, _, fscore = compute_metrics_on_validation_set(rnn, val_dataset)
ha4_part1_tests.test_min_f1_score(fscore)

---
### 1.6 Evaluation

Now that our model is trained, we can evaluate its predictions on the test set. 

In [ ]:
# Get all samples from the test set
x_test, y_test = test_dataset[:]

# Compute predictions
with torch.no_grad():
    test_out, _ = batch_forward_prop(rnn, x_test, y_test)

# Transform them into hard predictions
preds = test_out.argmax(dim=1)

# Get `preds` as a numpy array
preds = preds.cpu().numpy()

Using these predictions, we can compute the confusion matrix on the test set as follows.

In [ ]:
# Compute the confusion matrix
y_true = y_test.cpu().numpy()
cm = confusion_matrix(y_true, preds)
cm = cm.astype("float64")

# Normalize each row
cm /= cm.sum(axis=1, keepdims=True)

cm

In order to make this easier to visualize, let's plot it as a heat map:

In [ ]:
%matplotlib inline
plt.figure(figsize=(10, 10))
plt.imshow(cm)
plt.xticks(range(len(names_dict)), names_dict.keys(), rotation="vertical")
plt.yticks(range(len(names_dict)), names_dict.keys())
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.show()

\[extra\]: Why did we normalize the rows of the confusion matrix?

**Your answer:** (fill in here)

\[extra\]: What can you conclude from this confusion matrix? Which classes are easy/hard to classify? Which classes are being confused? 

**Your answer:** (fill in here)

To end this task, we can now define a function to perform predictions on any input name we provide (as long as the name contains only characters in our character set).

In [ ]:
def evaluate(name_tensor):
    with torch.no_grad():
        hidden = torch.zeros(1, rnn.hidden_size, device=device)
        for i in range(name_tensor.shape[1]):
            output, hidden = rnn(name_tensor[:, i, :], hidden)
    return output.exp()


def predict(input_line, charset, n_predictions=5):
    tensor = name2tensor(input_line, charset).to(device)[None, :]
    output = evaluate(tensor)

    # Get top N categories
    topv, topi = output.topk(n_predictions, 1, True)
    topv, topi = topv[0], topi[0]

    cats = [list(names_dict.keys())[i] for i in topi.cpu().numpy()]
    vs = topv.cpu().numpy()

    plt.figure(figsize=(10, 3))
    plt.bar(range(len(vs)), vs)
    plt.xticks(range(len(vs)), cats)

In [ ]:
predict("Åsakvi", charset)

In [ ]:
predict("Harakabim", charset)

In [ ]:
predict("Alakazam", charset)

In [ ]:
predict("Jin Quaio", charset)

In [ ]:
predict("Leonardino", charset)

In [ ]:
predict("Kim Sung", charset)

In [ ]:
predict("Thanos", charset)

In [ ]:
predict("Viiviika", charset)

Experiment on new names using the next cell: